방법:
동적이 있는 슬라이더를 구현하기 위해 plotly로 시각화를 한다.

내용:
슬라이더로 연간 탄소 배출량을 조절하도록 한다. (0~100 GtCO2 )<br>

연간 탄소 배출량에 따라 탄소 예산(250GtCO2)에서 지구의 평균 기온 상승 1.5℃에 도달하기까지<br> 예상되는 남은 년도와 월을 계산해 중간에 표시하고 남은 기간이 짧을 수록 <br> 바(bar)가 다시 원점에 가까워지게 하고 색은 붉은색에 가까워 지도록 한다.



In [12]:
import pandas as pd
import plotly.express as px
# 연간 탄소 배출량에 대한 자료 불러오기

df = pd.read_csv("co2-emissions-per-capita.csv")
df_1950 = df[df['Year'] >= 1950]    # 옛날 데이터는 없는경우도 많아서 1950년대를 기준으로 잡음


# 연간 탄소 배출량에 대한 증가량 시각화

# 국가별 색상, 한국어 이름 지정
highlight_countries = {
    'World': {'color': '#FF0000', 'ko_name': '전 세계 평균'},
    'United States': {'color': '#1F77B4', 'ko_name': '미국'},
    'China': {'color': '#FF7F0E', 'ko_name': '중국'},
    'India': {'color': '#2CA02C', 'ko_name': '인도'},
    'United Kingdom': {'color': '#9467BD', 'ko_name': '영국'},
    'South Korea': {'color': '#E377C2', 'ko_name': '한국'}
}


fig = px.line(
    df_1950, 
    x='Year', 
    y='CO₂ emissions per capita', 
    color='Entity',
    title='주요 국가별 1인당 CO2 배출량 추이 비교 (1950 ~ 현재)',
    labels={'CO₂ emissions per capita': '1인당 CO2 배출량 <span style="font-size:11px">(tCO₂)</span>'}
)

# 디자인 지정
for trace in fig.data:
    country_name = trace.name
    
    info = highlight_countries[country_name]
    ko_name = info['ko_name']
    
    trace.name = ko_name                   # 범례 이름을 한국어로 변경
    trace.line.color = info['color']       # 지정한 색상 부여
    trace.line.width = 3                   # 선 두께
    trace.opacity = 1.0                    # 투명도 
    trace.showlegend = True                # 범례 표시
    trace.mode = 'lines+markers'           # 선에 연도별 점(마커) 추가
    

    # Hover 텍스트
    trace.hovertemplate = (
        f"<b>{ko_name}</b><br>"
        f"연도: %{{x}}년<br>"
        f"배출량: %{{y:.2f}} tCO₂"
        f"<extra></extra>" 
    )
    


# 레이아웃 수정
fig.update_layout(
    plot_bgcolor='white',
    hovermode='closest',  # 개별 선에 hover 표시
    xaxis=dict(showgrid=True, gridcolor='lightgrey', title='연도 (Year)'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey'),
    legend_title_text='국가명'
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# 초기 설정값 
carbon_budget = 250.0  # 1.5도 방어를 위한 남은 탄소 예산 (GtCO2)
max_emission = 100.0   # 게이지의 최대치 (100 GtCO2)


emission_values = list(range(10, 101, 2)) # 슬라이더 배출량 
initial_emission = 40                     # 시작 시 기본 배출량

# 배출량에 따른 색상 결정 함수 
def get_color(val):
    if val <= 29:
        return '#2CA02C'  # 초록색 (안전)
    elif val <= 69:
        return '#FFC107'  # 노란색 (경고)
    else:
        return '#D62728'  # 빨간색 (위험)


# 원형 plot 구성
fig = go.Figure()

# 초기 시간에 계산
initial_years_left = carbon_budget / initial_emission
rem_y = int(initial_years_left)
rem_m = int((initial_years_left - rem_y) * 12)

# 도넛 차트를 활용한 원형 게이지
fig.add_trace(go.Pie(
    values=[initial_emission, max_emission - initial_emission],
    labels=['현재 배출량', '여유 공간'],
    marker=dict(colors=[get_color(initial_emission), '#E5E5E5']), # 채워진 곳 색상 + 빈 곳(회색)
    hole=0.8,          # 구멍크기, 크게 해서 시계 느낌 표현
    textinfo='none',   # 기본 도넛 텍스트 숨김
    hoverinfo='none',  # 호버 삭제
    sort=False,        # 값 크기에 따라 조각이 섞이지 않도록 고정
    direction='clockwise' # 시계 방향으로 차오르게 설정
))

# 초기 중앙 텍스트 설정
initial_text = (
    f"<span style='font-size:20px; color:gray'>남은 시간</span><br><br><br>"
    f"<b><span style='font-size:42px; color:black'>{rem_y}년 {rem_m}개월</span></b><br><br>"
    f"<span style='font-size:14px; color:#555'>현재 배출량: {initial_emission} GtCO₂/년</span>"
)


fig.update_layout(
    annotations=[dict(text=initial_text, x=0.5, y=0.5, showarrow=False)],
    title="<b>🌍 기후 위기 시계 (1.5℃ 도달 시점 예측)</b><br><sup>하단의 슬라이더를 움직여 연간 배출량을 조절해보세요.</sup>",
    title_x=0.5,
    showlegend=False,
    height=650
)


# 슬라이더에 연결할 기능

steps = []
for val in emission_values:
    # 슬라이더의 각 값(val)마다 남은 기한 새로 산정
    y_left = carbon_budget / val
    r_y = int(y_left)
    r_m = int((y_left - r_y) * 12)
    
    # 색상 갱신
    step_color = get_color(val)
    
    # 변경될 중앙 텍스트
    ann_text = (
        f"<span style='font-size:20px; color:gray'>남은 시간</span><br><br><br>"
        f"<b><span style='font-size:42px; color:{step_color}'>{r_y}년 {r_m}개월</span></b><br><br>"
        f"<span style='font-size:14px; color:#555'>예상 배출량: {val} GtCO₂/년</span>"
    )
    
    # 슬라이더 변경 시 업데이트 내용
    step = dict(
        method="update",
        args=[
            # 파이 차트의 값 비율과 색상 업데이트
            {"values": [[val, max_emission - val]], "marker.colors": [[step_color, '#E5E5E5']]},
            # 중앙 텍스트 내용 업데이트
            {"annotations": [dict(text=ann_text, x=0.5, y=0.5, showarrow=False)]}
        ],
        label=str(val) # 슬라이더 눈금에 보일 텍스트
    )
    steps.append(step)

# 슬라이더를 차트 하단에 배치
sliders = [dict(
    active=emission_values.index(initial_emission) if initial_emission in emission_values else 0,
    currentvalue={"prefix": "<b>조절된 예상 배출량: </b>", "suffix": " GtCO₂", "font": {"size": 18}},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(sliders=sliders)

fig.show()